# Qwen3 UltraChat-50k Results

This notebook reads saved training checkpoints and speculative-decoding eval summaries for the one-epoch UltraChat-50k Qwen3 sweeps. It does not load models, run training, or run evaluation.

Covered experiment pairs:

- `Qwen/Qwen3-14B` target with `Qwen/Qwen3-0.6B` draft
- `Qwen/Qwen3-8B` target with `Qwen/Qwen3-0.6B` draft

Expected artifacts are `checkpoints/<train_run>/meta.json`, optional HF trainer-state JSON files, and `/scratch/cs552-results/<eval_run>/eval_summary.json`.

## 1. Locate Artifacts

The defaults match the RunAI storage contract. Override `RESULTS_ROOT` or `CHECKPOINT_ROOT` in this cell if your artifacts live elsewhere.

In [10]:
from __future__ import annotations

import json
import math
import os
import re
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = print

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.chdir(REPO)

CHECKPOINT_ROOT = Path(os.environ.get("KDSD_CHECKPOINT_ROOT", REPO / "checkpoints")).expanduser()
RESULTS_ROOT = Path(os.environ.get("KDSD_RESULTS_ROOT", "/scratch/cs552-results")).expanduser()

DATA_ID = "ultrachat_50k"
SEED = 42
LOSSES = ("fkl", "rkl", "jsd", "ce")
EVAL_GAMMA = 4
EVAL_MAX_NEW_TOKENS = 256

print("repo root       :", REPO)
print("checkpoint root :", CHECKPOINT_ROOT)
print("results root    :", RESULTS_ROOT)
print("data id         :", DATA_ID)
print("seed            :", SEED)

repo root       : /scratch/cs552-repos/cs552-kdsd-youyang
checkpoint root : /scratch/cs552-repos/cs552-kdsd-youyang/checkpoints
results root    : /scratch/cs552-results
data id         : ultrachat_50k
seed            : 42


## 2. Experiment Set

These run-name patterns are the ones produced by `scripts/run_qwen3_loss_sweep.sh`, `scripts/submit_qwen3_loss_sweep.sh`, and `scripts/submit_qwen3_8b_loss_sweep.sh`.

In [11]:
EXPERIMENTS = [
    {
        "target_group": "Qwen3-14B target",
        "target": "Qwen/Qwen3-14B",
        "draft": "Qwen/Qwen3-0.6B",
        "train_prefix": "qwen3_0p6b",
        "eval_prefix": "qwen3_0p6b",
    },
    {
        "target_group": "Qwen3-8B target",
        "target": "Qwen/Qwen3-8B",
        "draft": "Qwen/Qwen3-0.6B",
        "train_prefix": "qwen3_8btarget_0p6b",
        "eval_prefix": "qwen3_8btarget_0p6b",
    },
]


def train_run_name(exp: dict[str, str], loss: str) -> str:
    return f"{exp['train_prefix']}_{loss}_{DATA_ID}_seed{SEED}"


def eval_run_name(exp: dict[str, str], loss_or_pretrain: str) -> str:
    return (
        f"{exp['eval_prefix']}_{loss_or_pretrain}_{DATA_ID}_seed{SEED}"
        f"_eval_g{EVAL_GAMMA}_max{EVAL_MAX_NEW_TOKENS}"
    )


for exp in EXPERIMENTS:
    print(exp["target_group"])
    print("  pretrained eval:", eval_run_name(exp, "pretrain"))
    for loss in LOSSES:
        print(" ", loss, "train:", train_run_name(exp, loss))
        print(" ", loss, "eval :", eval_run_name(exp, loss))

Qwen3-14B target
  pretrained eval: qwen3_0p6b_pretrain_ultrachat_50k_seed42_eval_g4_max256
  fkl train: qwen3_0p6b_fkl_ultrachat_50k_seed42
  fkl eval : qwen3_0p6b_fkl_ultrachat_50k_seed42_eval_g4_max256
  rkl train: qwen3_0p6b_rkl_ultrachat_50k_seed42
  rkl eval : qwen3_0p6b_rkl_ultrachat_50k_seed42_eval_g4_max256
  jsd train: qwen3_0p6b_jsd_ultrachat_50k_seed42
  jsd eval : qwen3_0p6b_jsd_ultrachat_50k_seed42_eval_g4_max256
  ce train: qwen3_0p6b_ce_ultrachat_50k_seed42
  ce eval : qwen3_0p6b_ce_ultrachat_50k_seed42_eval_g4_max256
Qwen3-8B target
  pretrained eval: qwen3_8btarget_0p6b_pretrain_ultrachat_50k_seed42_eval_g4_max256
  fkl train: qwen3_8btarget_0p6b_fkl_ultrachat_50k_seed42
  fkl eval : qwen3_8btarget_0p6b_fkl_ultrachat_50k_seed42_eval_g4_max256
  rkl train: qwen3_8btarget_0p6b_rkl_ultrachat_50k_seed42
  rkl eval : qwen3_8btarget_0p6b_rkl_ultrachat_50k_seed42_eval_g4_max256
  jsd train: qwen3_8btarget_0p6b_jsd_ultrachat_50k_seed42
  jsd eval : qwen3_8btarget_0p6b_jsd_ult

## 3. Load Helpers

In [12]:
def load_json(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        with path.open("r", encoding="utf-8") as fh:
            value = json.load(fh)
    except Exception as exc:
        return {"_load_error": f"{type(exc).__name__}: {exc}"}
    return value if isinstance(value, dict) else {"_load_error": "JSON root is not an object"}


def is_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value))


def fmt_number(value: Any, digits: int = 4) -> str:
    if value is None or value == "":
        return ""
    if is_number(value):
        return f"{float(value):.{digits}g}"
    return str(value)


def checkpoint_sort_key(path: Path) -> tuple[int, float]:
    match = re.search(r"checkpoint-(\d+)", str(path))
    step = int(match.group(1)) if match else -1
    try:
        mtime = path.stat().st_mtime
    except OSError:
        mtime = 0.0
    return step, mtime


def trainer_state_candidates(run_dir: Path) -> list[Path]:
    base = run_dir / "trainer_state"
    candidates = [base / "trainer_state.json"]
    if base.exists():
        candidates.extend(sorted(base.glob("checkpoint-*/trainer_state.json"), key=checkpoint_sort_key))
    return [path for path in candidates if path.exists()]


def load_latest_trainer_state(run_dir: Path) -> tuple[dict[str, Any] | None, Path | None]:
    candidates = trainer_state_candidates(run_dir)
    if not candidates:
        return None, None
    direct = run_dir / "trainer_state" / "trainer_state.json"
    path = direct if direct in candidates else candidates[-1]
    return load_json(path), path


def last_log_value(log_history: list[dict[str, Any]], key: str) -> Any:
    for record in reversed(log_history):
        if key in record and record[key] is not None:
            return record[key]
    return None


def first_existing_path(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def flatten_quality(summary: dict[str, Any] | None) -> dict[str, Any]:
    if not summary:
        return {}
    quality = summary.get("quality_score", {})
    if not isinstance(quality, dict):
        return {}
    return {f"quality:{key}": value for key, value in quality.items()}


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    if not rows:
        return "No rows."
    rendered = [[fmt_number(row.get(col)) for col in columns] for row in rows]
    widths = [len(col) for col in columns]
    for row in rendered:
        widths = [max(width, len(cell)) for width, cell in zip(widths, row)]
    header = "| " + " | ".join(col.ljust(width) for col, width in zip(columns, widths)) + " |"
    sep = "| " + " | ".join("-" * width for width in widths) + " |"
    body = ["| " + " | ".join(cell.ljust(width) for cell, width in zip(row, widths)) + " |" for row in rendered]
    return "\n".join([header, sep, *body])


def show_table(rows: list[dict[str, Any]], columns: list[str], *, title: str) -> None:
    print(title)
    if pd is not None:
        frame = pd.DataFrame(rows)
        for col in columns:
            if col not in frame.columns:
                frame[col] = None
        display(frame[columns])
    else:
        text = markdown_table(rows, columns)
        if Markdown is not None:
            display(Markdown(text))
        else:
            print(text)

## 4. Training Metrics

In [13]:
def training_row(exp: dict[str, str], loss: str) -> dict[str, Any]:
    run = train_run_name(exp, loss)
    run_dir = CHECKPOINT_ROOT / run
    meta_path = run_dir / "meta.json"
    config_path = run_dir / "config.yaml"
    model_dir = run_dir / "model"

    meta = load_json(meta_path)
    state, state_path = load_latest_trainer_state(run_dir)
    log_history = state.get("log_history", []) if isinstance(state, dict) else []
    if not isinstance(log_history, list):
        log_history = []

    status_bits = []
    if not meta_path.exists():
        status_bits.append("missing meta")
    if not model_dir.exists():
        status_bits.append("missing model")
    if state_path is None:
        status_bits.append("missing trainer_state")
    if isinstance(meta, dict) and meta.get("_load_error"):
        status_bits.append("meta load error")
    if isinstance(state, dict) and state.get("_load_error"):
        status_bits.append("trainer_state load error")

    meta = meta if isinstance(meta, dict) and "_load_error" not in meta else {}
    state = state if isinstance(state, dict) and "_load_error" not in state else {}

    return {
        "target_group": exp["target_group"],
        "target": meta.get("target", exp["target"]),
        "draft": meta.get("draft_init", exp["draft"]),
        "loss": loss,
        "train_run": run,
        "status": "; ".join(status_bits) if status_bits else "ok",
        "steps": meta.get("steps", state.get("global_step")),
        "epoch": last_log_value(log_history, "epoch"),
        "train_loss_final": meta.get("train_loss_final"),
        "last_logged_loss": last_log_value(log_history, "loss"),
        "last_eval_loss": last_log_value(log_history, "eval_loss"),
        "last_loss_ce": last_log_value(log_history, "loss_ce"),
        "last_loss_kd": last_log_value(log_history, "loss_kd"),
        "learning_rate": last_log_value(log_history, "learning_rate"),
        "checkpoint_dir": str(run_dir),
        "trainer_state": str(state_path) if state_path else "",
        "config_exists": config_path.exists(),
    }


training_rows = [training_row(exp, loss) for exp in EXPERIMENTS for loss in LOSSES]

training_columns = [
    "target_group",
    "loss",
    "status",
    "steps",
    "epoch",
    "train_loss_final",
    "last_logged_loss",
    "last_eval_loss",
    "last_loss_ce",
    "last_loss_kd",
    "learning_rate",
    "train_run",
]

show_table(training_rows, training_columns, title="Training summary")

Training summary


,target_group,loss,status,steps,epoch,train_loss_final,last_logged_loss,last_eval_loss,last_loss_ce,last_loss_kd,learning_rate,train_run
0,Qwen3-14B target,fkl,ok,0,0.998206,0.586793,0.5748,0.574336,1.564736,0.574782,1.785399e-10,qwen3_0p6b_fkl_ultrachat_50k_seed42
1,Qwen3-14B target,rkl,ok,0,0.998206,0.788214,0.7792,0.755836,1.883481,0.779212,1.785399e-10,qwen3_0p6b_rkl_ultrachat_50k_seed42
2,Qwen3-14B target,jsd,ok,0,0.998206,0.107922,0.1062,0.105777,1.697493,0.106192,1.785399e-10,qwen3_0p6b_jsd_ultrachat_50k_seed42
3,Qwen3-14B target,ce,ok,0,0.998206,1.340084,1.2896,1.318011,1.289562,0.000000,1.785399e-10,qwen3_0p6b_ce_ultrachat_50k_seed42
4,Qwen3-8B target,fkl,ok,0,0.998206,0.557518,0.5527,0.538360,1.562143,0.552665,1.785399e-10,qwen3_8btarget_0p6b_fkl_ultrachat_50k_seed42
5,Qwen3-8B target,rkl,ok,0,0.998206,0.779047,0.7659,0.742144,1.900097,0.765932,1.785399e-10,qwen3_8btarget_0p6b_rkl_ultrachat_50k_seed42
6,Qwen3-8B target,jsd,ok,0,0.998206,0.105123,0.1037,0.101929,1.699414,0.103747,1.785399e-10,qwen3_8btarget_0p6b_jsd_ultrachat_50k_seed42
7,Qwen3-8B target,ce,ok,0,0.998206,1.393722,1.3457,1.353036,1.345719,0.000000,1.785399e-10,qwen3_8btarget_0p6b_ce_ultrachat_50k_seed42


## 5. Speculative-Decoding Eval Metrics

In [14]:
def eval_row(exp: dict[str, str], loss_or_pretrain: str) -> dict[str, Any]:
    run = eval_run_name(exp, loss_or_pretrain)
    result_dir = RESULTS_ROOT / run
    summary_path = result_dir / "eval_summary.json"
    timing_path = result_dir / "timing.json"
    config_path = result_dir / "config.yaml"
    generations_path = result_dir / "generations.jsonl"

    summary = load_json(summary_path)
    status_bits = []
    if not summary_path.exists():
        status_bits.append("missing eval_summary")
    if not timing_path.exists():
        status_bits.append("missing timing")
    if isinstance(summary, dict) and summary.get("_load_error"):
        status_bits.append("summary load error")

    summary = summary if isinstance(summary, dict) and "_load_error" not in summary else {}
    decoding = summary.get("decoding", {}) if isinstance(summary.get("decoding", {}), dict) else {}
    engines = summary.get("engines", {}) if isinstance(summary.get("engines", {}), dict) else {}
    hf_engine = engines.get("hf", {}) if isinstance(engines.get("hf", {}), dict) else {}

    row = {
        "target_group": exp["target_group"],
        "target": summary.get("target", exp["target"]),
        "draft": summary.get("draft", exp["draft"]),
        "loss": loss_or_pretrain,
        "run_kind": "pretrained" if loss_or_pretrain == "pretrain" else "trained",
        "eval_run": run,
        "status": "; ".join(status_bits) if status_bits else "ok",
        "acceptance_rate": summary.get("acceptance_rate"),
        "avg_accepted_tokens": summary.get("avg_accepted_tokens"),
        "speedup": summary.get("speedup"),
        "tokens_per_second": summary.get("tokens_per_second"),
        "sd_time_s": summary.get("sd_time_s"),
        "vanilla_time_s": summary.get("vanilla_time_s"),
        "n_prompts": summary.get("n_prompts"),
        "n_warmup": summary.get("n_warmup"),
        "n_repeats": summary.get("n_repeats"),
        "gamma": decoding.get("num_assistant_tokens", decoding.get("gamma")),
        "max_new_tokens": decoding.get("max_new_tokens"),
        "decoding_mode": decoding.get("mode"),
        "n_outer_steps": hf_engine.get("n_outer_steps"),
        "target_calls": hf_engine.get("target_calls"),
        "draft_calls": hf_engine.get("draft_calls"),
        "draft_forward_s": hf_engine.get("draft_forward_s"),
        "target_forward_s": hf_engine.get("target_forward_s"),
        "result_dir": str(result_dir),
        "config_exists": config_path.exists(),
        "generations_exists": generations_path.exists(),
    }
    row.update(flatten_quality(summary))
    return row


eval_rows = []
for exp in EXPERIMENTS:
    eval_rows.append(eval_row(exp, "pretrain"))
    for loss in LOSSES:
        eval_rows.append(eval_row(exp, loss))


def add_baseline_deltas(rows: list[dict[str, Any]]) -> None:
    baselines = {
        row["target_group"]: row
        for row in rows
        if row.get("run_kind") == "pretrained" and row.get("status") == "ok"
    }
    for row in rows:
        base = baselines.get(row["target_group"])
        if not base or row.get("run_kind") == "pretrained":
            row["speedup_delta_vs_pretrain"] = None
            row["acceptance_delta_vs_pretrain"] = None
            continue
        row["speedup_delta_vs_pretrain"] = (
            row.get("speedup") - base.get("speedup")
            if is_number(row.get("speedup")) and is_number(base.get("speedup"))
            else None
        )
        row["acceptance_delta_vs_pretrain"] = (
            row.get("acceptance_rate") - base.get("acceptance_rate")
            if is_number(row.get("acceptance_rate")) and is_number(base.get("acceptance_rate"))
            else None
        )


def add_best_flags(rows: list[dict[str, Any]]) -> None:
    for target_group in sorted({row["target_group"] for row in rows}):
        group_rows = [row for row in rows if row["target_group"] == target_group and row.get("status") == "ok"]
        speedups = [float(row["speedup"]) for row in group_rows if is_number(row.get("speedup"))]
        accepts = [float(row["acceptance_rate"]) for row in group_rows if is_number(row.get("acceptance_rate"))]
        best_speedup = max(speedups) if speedups else None
        best_accept = max(accepts) if accepts else None
        for row in rows:
            if row["target_group"] != target_group:
                continue
            row["best_speedup_in_target_group"] = bool(
                best_speedup is not None and is_number(row.get("speedup")) and math.isclose(float(row["speedup"]), best_speedup)
            )
            row["best_acceptance_in_target_group"] = bool(
                best_accept is not None and is_number(row.get("acceptance_rate")) and math.isclose(float(row["acceptance_rate"]), best_accept)
            )


add_baseline_deltas(eval_rows)
add_best_flags(eval_rows)

quality_columns = sorted({key for row in eval_rows for key in row if key.startswith("quality:")})
eval_columns = [
    "target_group",
    "loss",
    "run_kind",
    "status",
    "acceptance_rate",
    "avg_accepted_tokens",
    "speedup",
    "speedup_delta_vs_pretrain",
    "acceptance_delta_vs_pretrain",
    "tokens_per_second",
    "sd_time_s",
    "vanilla_time_s",
    "n_prompts",
    "gamma",
    "max_new_tokens",
    "best_speedup_in_target_group",
    "best_acceptance_in_target_group",
    *quality_columns,
    "eval_run",
]

show_table(eval_rows, eval_columns, title="Speculative-decoding eval summary")

Speculative-decoding eval summary


,target_group,loss,run_kind,status,acceptance_rate,avg_accepted_tokens,speedup,speedup_delta_vs_pretrain,acceptance_delta_vs_pretrain,tokens_per_second,sd_time_s,vanilla_time_s,n_prompts,gamma,max_new_tokens,best_speedup_in_target_group,best_acceptance_in_target_group,eval_run
0,Qwen3-14B target,pretrain,pretrained,ok,0.377279,1.509117,0.452167,NaN,NaN,11.553634,1107.270699,500.671179,50,4,256,False,False,qwen3_0p6b_pretrain_ultrachat_50k_seed42_eval_...
1,Qwen3-14B target,fkl,trained,ok,0.353581,1.414324,0.433118,-0.019049,-0.023698,10.900977,1174.114962,508.529893,50,4,256,False,False,qwen3_0p6b_fkl_ultrachat_50k_seed42_eval_g4_ma...
2,Qwen3-14B target,rkl,trained,ok,0.401794,1.607177,0.466921,0.014754,0.024515,11.932226,1070.965305,500.056605,50,4,256,True,True,qwen3_0p6b_rkl_ultrachat_50k_seed42_eval_g4_ma...
3,Qwen3-14B target,jsd,trained,ok,0.391866,1.567465,0.461889,0.009722,0.014587,11.604634,1101.025703,508.551782,50,4,256,False,False,qwen3_0p6b_jsd_ultrachat_50k_seed42_eval_g4_ma...
4,Qwen3-14B target,ce,trained,ok,0.317773,1.271093,0.409639,-0.042528,-0.059506,10.442012,1224.668184,501.671419,50,4,256,False,False,qwen3_0p6b_ce_ultrachat_50k_seed42_eval_g4_max256
5,Qwen3-8B target,pretrain,pretrained,ok,0.401702,1.606810,0.435445,NaN,NaN,12.521989,1022.201805,445.112758,50,4,256,False,False,qwen3_8btarget_0p6b_pretrain_ultrachat_50k_see...
6,Qwen3-8B target,fkl,trained,ok,0.383306,1.533224,0.423789,-0.011656,-0.018396,12.234384,1046.231688,443.381522,50,4,256,False,False,qwen3_8btarget_0p6b_fkl_ultrachat_50k_seed42_e...
7,Qwen3-8B target,rkl,trained,ok,0.420461,1.681843,0.447302,0.011857,0.018758,12.341437,1037.156334,463.921787,50,4,256,True,True,qwen3_8btarget_0p6b_rkl_ultrachat_50k_seed42_e...
8,Qwen3-8B target,jsd,trained,ok,0.417861,1.671444,0.445483,0.010038,0.016159,12.575287,1017.789927,453.407748,50,4,256,False,False,qwen3_8btarget_0p6b_jsd_ultrachat_50k_seed42_e...
9,Qwen3-8B target,ce,trained,ok,0.331139,1.324556,0.388792,-0.046653,-0.070564,11.352237,1127.531054,438.375082,50,4,256,False,False,qwen3_8btarget_0p6b_ce_ultrachat_50k_seed42_ev...


## 6. Cross-Target Comparison

This view aligns the same loss across the 14B-target and 8B-target sweeps.

In [15]:
def row_by_target_and_loss(rows: list[dict[str, Any]], target_group: str, loss: str) -> dict[str, Any] | None:
    for row in rows:
        if row.get("target_group") == target_group and row.get("loss") == loss:
            return row
    return None


comparison_rows = []
for loss in ("pretrain", *LOSSES):
    row_14b = row_by_target_and_loss(eval_rows, "Qwen3-14B target", loss)
    row_8b = row_by_target_and_loss(eval_rows, "Qwen3-8B target", loss)
    comparison_rows.append(
        {
            "loss": loss,
            "14b_status": row_14b.get("status") if row_14b else "missing row",
            "8b_status": row_8b.get("status") if row_8b else "missing row",
            "14b_acceptance": row_14b.get("acceptance_rate") if row_14b else None,
            "8b_acceptance": row_8b.get("acceptance_rate") if row_8b else None,
            "acceptance_8b_minus_14b": (
                row_8b.get("acceptance_rate") - row_14b.get("acceptance_rate")
                if row_14b and row_8b and is_number(row_14b.get("acceptance_rate")) and is_number(row_8b.get("acceptance_rate"))
                else None
            ),
            "14b_speedup": row_14b.get("speedup") if row_14b else None,
            "8b_speedup": row_8b.get("speedup") if row_8b else None,
            "speedup_8b_minus_14b": (
                row_8b.get("speedup") - row_14b.get("speedup")
                if row_14b and row_8b and is_number(row_14b.get("speedup")) and is_number(row_8b.get("speedup"))
                else None
            ),
        }
    )

comparison_columns = [
    "loss",
    "14b_status",
    "8b_status",
    "14b_acceptance",
    "8b_acceptance",
    "acceptance_8b_minus_14b",
    "14b_speedup",
    "8b_speedup",
    # "speedup_8b_minus_14b",
]

show_table(comparison_rows, comparison_columns, title="8B-target minus 14B-target comparison")

8B-target minus 14B-target comparison


,loss,14b_status,8b_status,14b_acceptance,8b_acceptance,acceptance_8b_minus_14b,14b_speedup,8b_speedup
0,pretrain,ok,ok,0.377279,0.401702,0.024423,0.452167,0.435445
1,fkl,ok,ok,0.353581,0.383306,0.029725,0.433118,0.423789
2,rkl,ok,ok,0.401794,0.420461,0.018667,0.466921,0.447302
3,jsd,ok,ok,0.391866,0.417861,0.025995,0.461889,0.445483
4,ce,ok,ok,0.317773,0.331139,0.013366,0.409639,0.388792


## 7. Conclusions From Loaded Results

1. Training dynamics: Both experimented models are trained with descent loss decrease and evaluation loss decrease. But for 14b training, the training loss and evaluation loss is generally higher than 8b. This is because for the same small draft model, larger target model is harder to imitate.

2. Speedup comparsion: Both result the same speedup order from high to low: $$\text{RKL} \approx \text{JSD} > \text{Pretrained} > \text{FKL} > \text{CE}$$
So the loss sweep comparisons are consistent among different model selections, which indicates the potential of JSD and RKL loss.
For cross-model results, using 0.6b draft for 14b target has lower acceptance rate than 8b target, which aligns well as the training dynamics conclusion above.

3. Acceptance rate comparison: it follows the same order as the speedup. But for cross-model results, we find that even if 14b generally suffers from low acceptance rate, it gains better speedup because the baseline inference speed is lower due to larger target size.

## 8. Artifact Paths

Use this cell when a row is marked missing.

In [16]:
path_rows = []
for row in training_rows:
    path_rows.append(
        {
            "kind": "train",
            "target_group": row["target_group"],
            "loss": row["loss"],
            "status": row["status"],
            "path": row["checkpoint_dir"],
        }
    )
for row in eval_rows:
    path_rows.append(
        {
            "kind": "eval",
            "target_group": row["target_group"],
            "loss": row["loss"],
            "status": row["status"],
            "path": row["result_dir"],
        }
    )

show_table(path_rows, ["kind", "target_group", "loss", "status", "path"], title="Resolved artifact paths")

Resolved artifact paths


,kind,target_group,loss,status,path
0,train,Qwen3-14B target,fkl,ok,/scratch/cs552-repos/cs552-kdsd-youyang/checkp...
1,train,Qwen3-14B target,rkl,ok,/scratch/cs552-repos/cs552-kdsd-youyang/checkp...
2,train,Qwen3-14B target,jsd,ok,/scratch/cs552-repos/cs552-kdsd-youyang/checkp...
3,train,Qwen3-14B target,ce,ok,/scratch/cs552-repos/cs552-kdsd-youyang/checkp...
4,train,Qwen3-8B target,fkl,ok,/scratch/cs552-repos/cs552-kdsd-youyang/checkp...
5,train,Qwen3-8B target,rkl,ok,/scratch/cs552-repos/cs552-kdsd-youyang/checkp...
6,train,Qwen3-8B target,jsd,ok,/scratch/cs552-repos/cs552-kdsd-youyang/checkp...
7,train,Qwen3-8B target,ce,ok,/scratch/cs552-repos/cs552-kdsd-youyang/checkp...
8,eval,Qwen3-14B target,pretrain,ok,/scratch/cs552-results/qwen3_0p6b_pretrain_ult...
9,eval,Qwen3-14B target,fkl,ok,/scratch/cs552-results/qwen3_0p6b_fkl_ultracha...
